In [5]:
# Install jika belum ada
!pip install scikit-learn

# Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
from google.colab import drive
drive.mount('/content/drive')

# Memuat dataset CSV
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/dicoding #9/playstore_reviews.csv')

# Tampilkan beberapa baris pertama
df.head()



print("Jumlah data:", len(df))
df.head()

# The column name for rating is 'score'
rating_column_name = 'score'

# Label sentimen dari skor rating
def label_sentiment(score):
    if score >= 4:
        return 'positif'
    elif score == 3:
        return 'netral'
    else:
        return 'negatif'

# Apply the function to the correct column
df['sentiment'] = df[rating_column_name].apply(label_sentiment)

# Cek distribusi label
print("Distribusi sentimen:\n", df['sentiment'].value_counts())

# TF-IDF untuk ekstraksi fitur
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['content'].astype(str)).toarray() #Change 'review' to 'content'

# Encode label ke angka
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])  # 0=negatif, 1=netral, 2=positif

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Latih model Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n🎯 Akurasi Testing: {acc * 100:.2f}%")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# Inference (Prediksi Kalimat Baru)
def prediksi_sentimen(kalimat):
    vector = vectorizer.transform([kalimat]).toarray()
    hasil = model.predict(vector)
    return le.inverse_transform(hasil)[0]

# Contoh prediksi
contoh = "Aplikasi ini sangat jelek dan sering error"
hasil = prediksi_sentimen(contoh)
print("\n🧠 Inference Contoh:")
print(f"Kalimat: '{contoh}'")
print(f"Hasil Prediksi Sentimen: {hasil}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah data: 12495
Distribusi sentimen:
 sentiment
positif    5654
negatif    4850
netral     1991
Name: count, dtype: int64

🎯 Akurasi Testing: 71.39%

📊 Classification Report:
               precision    recall  f1-score   support

     negatif       0.66      0.85      0.74       970
      netral       0.25      0.01      0.02       398
     positif       0.77      0.85      0.81      1131

    accuracy                           0.71      2499
   macro avg       0.56      0.57      0.52      2499
weighted avg       0.65      0.71      0.66      2499


🧠 Inference Contoh:
Kalimat: 'Aplikasi ini sangat jelek dan sering error'
Hasil Prediksi Sentimen: negatif
